# Fechas, Manejo de Errores y Macros en Rust

Este notebook cubre tres áreas fundamentales para el desarrollo de aplicaciones robustas y eficientes en Rust:
1. **Manejo de Fechas y Tiempo** (usando la librería estándar y `chrono`).
2. **Estrategias avanzadas de Manejo de Errores**.
3. **Creación de Macros** (Metaprogramación).

## 1. Manejo de Fechas y Tiempo

Rust proporciona `std::time` para mediciones básicas de tiempo (duración, instantes), pero para manejo de calendarios y zonas horarias, la comunidad utiliza casi universalmente la crate `chrono`.

Para usar `chrono` en este notebook (kernel `evcxr`), primero debemos añadir la dependencia.

In [3]:
:dep chrono = "0.4"

In [4]:
use chrono::prelude::*;
use std::time::{Duration, Instant};

// --- Medir duración básica (std::time) ---
let ahora = Instant::now();
// Simulación de carga
let diez_millis = Duration::from_millis(10);
std::thread::sleep(diez_millis);

let transcurrido = ahora.elapsed();
println!("Tiempo transcurrido: {:?}", transcurrido);

// --- Manejo de Fechas (Chrono) ---
let utc_ahora: DateTime<Utc> = Utc::now();
println!("UTC: {}", utc_ahora);

let local_ahora: DateTime<Local> = Local::now();
println!("Local: {}", local_ahora);

// Formateo de fechas
println!("Formateado: {}", local_ahora.format("%Y-%m-%d %H:%M:%S"));

// Parseo de fechas
let fecha_str = "2023-10-27 15:30:00";
let fecha_parseada = NaiveDateTime::parse_from_str(fecha_str, "%Y-%m-%d %H:%M:%S");

match fecha_parseada {
    Ok(dt) => println!("Fecha parseada correctamente: {}", dt),
    Err(e) => println!("Error al parsear: {}", e),
}

Tiempo transcurrido: 10.056771ms
UTC: 2026-03-04 11:34:23.063978149 UTC
Local: 2026-03-04 11:34:23.063986034 +00:00
Formateado: 2026-03-04 11:34:23
Fecha parseada correctamente: 2023-10-27 15:30:00


()

## 2. Manejo de Errores Avanzado

En Rust, el manejo de errores se basa en los tipos `Option<T>` y `Result<T, E>`. Para aplicaciones reales, lo ideal es definir tipos de error propios que implementen el trait `std::error::Error`.

In [5]:
use std::fmt;
use std::io;

// Definición de nuestro propio tipo de Error
#[derive(Debug)]
enum MiError {
    Io(io::Error),
    FormatoInvalido(String),
    DatoNoEncontrado,
}

// Implementar Display para nuestro error
impl fmt::Display for MiError {
    fn fmt(&self, f: &mut fmt::Formatter) -> fmt::Result {
        match self {
            MiError::Io(err) => write!(f, "Error de I/O: {}", err),
            MiError::FormatoInvalido(msg) => write!(f, "Formato inválido: {}", msg),
            MiError::DatoNoEncontrado => write!(f, "Dato no encontrado en el sistema"),
        }
    }
}

// Implementar el trait Error
impl std::error::Error for MiError {}

// Permitir la conversión automática de io::Error a MiError usando el operador ?
impl From<io::Error> for MiError {
    fn from(err: io::Error) -> MiError {
        MiError::Io(err)
    }
}

fn ejemplo_error(simular_fallo: bool) -> Result<String, MiError> {
    if simular_fallo {
        return Err(MiError::DatoNoEncontrado);
    }
    Ok(String::from("Éxito"))
}

match ejemplo_error(true) {
    Ok(s) => println!("Resultado: {}", s),
    Err(e) => println!("Ocurrió un error: {}", e),
}

Ocurrió un error: Dato no encontrado en el sistema


()

## 3. Creación de Macros

Las macros en Rust son una forma de escribir código que genera código.

In [6]:
// Una macro que crea un vector y añade elementos automáticamente
macro_rules! crear_lista {
    ( $( $x:expr ),* ) => {
        {
            let mut temp_vec = Vec::new();
            $(
                temp_vec.push($x);
            )*
            temp_vec
        }
    };
}

let mi_lista = crear_lista!(1, 2, 3, 4, 5);
println!("Lista creada por macro: {:?}", mi_lista);

// Macro para depuración rápida
macro_rules! log_val {
    ($val:expr) => {
        println!("{} = {:?}", stringify!($val), &$val);
    };
}

let x = 42;
log_val!(x);

Lista creada por macro: [1, 2, 3, 4, 5]
x = 42
